In [ ]:
!pip uninstall mediapipe protobuf -y
!pip install mediapipe==0.10.14 protobuf==4.25.3 "numpy<2" tensorflow scikit-learn

Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
LETTER_PATH = "/content/drive/MyDrive/asl_letters"

In [ ]:
import os
import cv2
import random
import pickle
import numpy as np
import mediapipe as mp
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# =========================================
# PATH
# =========================================
LETTER_PATH = "/content/drive/MyDrive/asl_letters"

# =========================================
# MEDIAPIPE
# =========================================
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)

# =========================================
# DATA COLLECTION
# =========================================
data = []
labels = []

MAX_IMAGES = 300

print("🚀 Starting preprocessing...")

for class_dir in os.listdir(LETTER_PATH):

    path = os.path.join(LETTER_PATH, class_dir)

    if not os.path.isdir(path):
        continue

    print(f"\n📁 Processing: {class_dir}")

    images = os.listdir(path)
    random.shuffle(images)

    count = 0

    for img_name in images:

        if count >= MAX_IMAGES:
            break

        img_path = os.path.join(path, img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        results = hands.process(img_rgb)

        if results.multi_hand_landmarks:

            for hand_landmarks in results.multi_hand_landmarks:

                row = []

                for lm in hand_landmarks.landmark:
                    row.append(lm.x)
                    row.append(lm.y)

                # Must be exactly 42
                if len(row) == 42:

                    data.append(row)
                    labels.append(class_dir)

                    count += 1

        if count % 50 == 0 and count > 0:
            print(f"   {count} images processed")

print("\n💾 Saving dataset...")

data = np.array(data)
labels = np.array(labels)

np.save("letter_data.npy", data)
np.save("letter_labels.npy", labels)

print("✅ Dataset saved")
print("Samples:", len(data))

# =========================================
# ENCODING
# =========================================
print("\n🔤 Encoding labels...")

le = LabelEncoder()

labels_enc = le.fit_transform(labels)

labels_cat = to_categorical(labels_enc)

with open("letter_encoder.pickle", "wb") as f:
    pickle.dump(le, f)

# =========================================
# RESHAPE
# =========================================
data = data.reshape(len(data), 1, 42)

# =========================================
# SPLIT
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    data,
    labels_cat,
    test_size=0.2,
    random_state=42,
    stratify=labels_enc
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# =========================================
# MODEL
# =========================================
print("\n🧠 Building model...")

model = Sequential([

    tf.keras.Input(shape=(1, 42)),

    LSTM(64),

    Dropout(0.3),

    Dense(64, activation='relu'),

    Dense(32, activation='relu'),

    Dense(len(le.classes_), activation='softmax')
])

# =========================================
# COMPILE
# =========================================
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =========================================
# CALLBACKS
# =========================================
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

# =========================================
# TRAIN
# =========================================
print("\n🚀 Training started...")

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)

# =========================================
# EVALUATE
# =========================================
loss, acc = model.evaluate(X_test, y_test)

print(f"\n✅ Accuracy: {acc:.4f}")

# =========================================
# SAVE MODEL
# =========================================
model.save(
    "letter_model.h5",
    include_optimizer=False
)

print("\n✅ Letter model saved successfully")

🚀 Starting preprocessing...

📁 Processing: Z


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


   50 images processed
   100 images processed
   150 images processed
   200 images processed
   250 images processed
   300 images processed

📁 Processing: Y
   50 images processed
   100 images processed
   150 images processed
   200 images processed
   250 images processed
   300 images processed

📁 Processing: X
   50 images processed
   100 images processed
   150 images processed
   200 images processed
   250 images processed
   250 images processed
   250 images processed
   300 images processed

📁 Processing: V
   50 images processed
   100 images processed
   150 images processed
   200 images processed
   250 images processed
   300 images processed

📁 Processing: U
   50 images processed
   50 images processed
   100 images processed
   150 images processed
   200 images processed
   200 images processed
   200 images processed
   250 images processed
   300 images processed

📁 Processing: W
   50 images processed
   100 images processed
   150 images processed
   200 ima

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        27,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 26)             │           858 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,490 (134.73 KB)

 Trainable params: 34,490 (134.73 KB)

 Non-trainable params: 0 (0.00 B)


🚀 Training started...
Epoch 1/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.1163 - loss: 3.0975 - val_accuracy: 0.3109 - val_loss: 2.5015
Epoch 2/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4981 - loss: 1.7122 - val_accuracy: 0.7551 - val_loss: 1.0240
Epoch 3/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7322 - loss: 0.9364 - val_accuracy: 0.7929 - val_loss: 0.7346
Epoch 4/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7787 - loss: 0.7309 - val_accuracy: 0.8538 - val_loss: 0.5413
Epoch 5/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8170 - loss: 0.6006 - val_accuracy: 0.8788 - val_loss: 0.4672
Epoch 6/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8218 - loss: 0.5604 - val_accuracy: 0.8673 - val_loss: 0.4440
Epoch 7/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8361 - loss: 0.5100 - val_accuracy: 0.8981 - val_loss: 0.4054
Epoch 8/30
195/195 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8518 - loss: 0.


✅ Accuracy: 0.9558

✅ Letter model saved successfully
